# SHIPP Stage 9B — Databricks AI Search index + semantic probe

`gold_listing_search_docs` → Vector Search endpoint `VS_ENDPOINT_NAME` → Delta Sync index `VS_INDEX_NAME`

| Decision | Value |
|---|---|
| Index type | Delta Sync, TRIGGERED (sync after every Gold MERGE — measured below) |
| Primary key | `listing_id` |
| Embedded column | `search_text`, embedded by `EMBEDDING_ENDPOINT` (Databricks-managed embeddings) |
| Proof | a query that is **not** the listing title returns the expected listing, and every result traces to Gold |

Creates the endpoint/index only if missing. **Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert.

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
# Reload the rag modules so edits to rag/*.py always take effect (same rule as shipp_silver_lib).
for _name in ["rag.text", "rag.vision", "rag.content", "rag.search_docs", "rag.index",
              "rag.retrieval", "rag.semantic", "rag.evaluation"]:
    importlib.reload(importlib.import_module(_name))

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("RAG modules reloaded from", REPO_ROOT)

In [ ]:
try:
    _ep = w.serving_endpoints.get(EMBEDDING_ENDPOINT)
    print("Embedding endpoint:", _ep.name, "| ready:", _ep.state.ready if _ep.state else None)
except Exception as e:
    _names = sorted(s.name for s in w.serving_endpoints.list())
    raise AssertionError(
        f"FAILED: embedding endpoint {EMBEDDING_ENDPOINT!r} not found ({type(e).__name__}). "
        f"Set EMBEDDING_ENDPOINT in config/settings.py to one of: {_names}"
    )

## Step 1 — Endpoint + index

In [ ]:
from rag import index
from rag.retrieval import search_listing_context

state = index.ensure_endpoint(w, VS_ENDPOINT_NAME)
assert state == "ONLINE", f"FAILED: Vector Search endpoint {VS_ENDPOINT_NAME} is {state}, not ONLINE"

columns_to_sync = SEARCH_RETURN_COLUMNS + ["vision_status", "document_updated_at"]
created = index.ensure_delta_sync_index(
    w, VS_INDEX_NAME, VS_ENDPOINT_NAME, GOLD_LISTING_SEARCH_DOCS,
    primary_key="listing_id", text_column=SEARCH_TEXT_COLUMN,
    embedding_endpoint=EMBEDDING_ENDPOINT, columns_to_sync=columns_to_sync,
)

## Step 2 — Sync and measure how long Gold → searchable takes

In [ ]:
expected_rows = spark.table(GOLD_LISTING_SEARCH_DOCS).count()
if not created:
    try:
        index.trigger_sync(w, VS_INDEX_NAME)
        print("Sync triggered.")
    except Exception as e:  # a sync already running is fine; we wait for it below
        print(f"Sync not triggered ({type(e).__name__}: {str(e)[:150]}) — waiting for the running one.")
sync_seconds = index.wait_until_synced(w, VS_INDEX_NAME, expected_rows)
print(f"PASS — index holds {expected_rows} docs. Gold → searchable: {sync_seconds}s")
print(index.index_status(w, VS_INDEX_NAME))

## Step 3 — Semantic probe with traceability

In [ ]:
t0 = time.time()
results = search_listing_context(RAG_PROBE_QUERY, k=3, workspace_client=w)
probe_ms = int((time.time() - t0) * 1000)

print(f"Query: {RAG_PROBE_QUERY!r}  ({probe_ms} ms)")
for r in results:
    print(f"  {(r['score'] or 0):.4f}  {r['listing_id']:<20} {r['title']}  | {r['snippet'][:120]}")

gold_ids = {r.listing_id for r in spark.table(GOLD_LISTING_SEARCH_DOCS).select("listing_id").collect()}
assert results, "FAILED: the probe returned nothing."
assert all(r["listing_id"] in gold_ids for r in results), "FAILED: a result does not trace to Gold"
assert RAG_PROBE_EXPECTED_LISTING in [r["listing_id"] for r in results], (
    f"FAILED: {RAG_PROBE_EXPECTED_LISTING} not in the top 3 — check it is AVAILABLE and in Gold."
)
print("PASS — semantic retrieval returns the expected listing, every result traceable by listing_id.")

summary = {"index": VS_INDEX_NAME, "docs": expected_rows, "sync_seconds": sync_seconds,
           "probe_latency_ms": probe_ms, "status": "PASS"}
print(summary)
dbutils.notebook.exit(json.dumps(summary))